In [1]:
#Library
# ---
# Librerias de selenium 

from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import Select  # selccionar opciones de las lista desplegable
from selenium.webdriver.common.by import By  # permite seleccionar los elementos en un html
from selenium.webdriver.common.keys import Keys  # ingresar datos a un formulario
from selenium.common.exceptions import NoSuchElementException
# Opciones driver
from selenium.webdriver.chrome.options import Options
from seleniumbase import Driver

# dataframes
import pandas as pd
from pandas import json_normalize
import numpy as np

# Similamos comportamiento humano al añadir tiempos aleatorios
import time
from time import sleep
import random
from tqdm import tqdm

import requests 

# Manipular Json files
import json

from utils import * 

In [2]:
# Open la página

options_chrome = Options()
options_chrome.add_argument('--disable-blink-features=AutomationControlled')

driver = Driver(browser="chrome", headless=False)



# Access to INDECOPI website - Denuncias de ciudadanos
url = 'https://servicio.indecopi.gob.pe/buscadorResoluciones/proteccion-consumidor.seam'
driver.get(url)
driver.maximize_window()

In [3]:
web_filter_consult = driver.find_element(By.XPATH, "//input[@id='FormListado1:txtTextoBusqueda']")

web_filter_consult.send_keys('Fraude financiero en bancos')


In [4]:
driver.find_element(By.XPATH, "//input[@id='FormListado1:b_RealizaBusqueda']").click()

In [ ]:
# links_pdf = driver.find_elements(By.XPATH, "//span[contains(@onclick,'descargar')]")

In [ ]:
num_pages = 7

data_full = {}

In [15]:
for i in tqdm(range(num_pages)):
    
    nro_resoluciones = []
    fecha_resoluciones = []
    exp_resoluciones = []
    denuncias = []
    summillas = []

    lsit_1 = driver.find_elements(By.XPATH, "//td[contains(@id, 'id134')]")
    nro_resoluciones.extend([x.text for x in lsit_1])
    
    lsit_2 = driver.find_elements(By.XPATH, "//td[contains(@id, 'id138')]")
    fecha_resoluciones.extend([x.text for x in lsit_2])

    lsit_3 = driver.find_elements(By.XPATH, "//td[contains(@id, 'id140')]")
    exp_resoluciones.extend([x.text for x in lsit_3])

    lsit_4 = driver.find_elements(By.XPATH, "//td[contains(@id, 'id142')]")
    denuncias.extend([x.text for x in lsit_4])

    lsit_5 = driver.find_elements(By.XPATH, "//td[contains(@id, 'sumilla')]")
    summillas.extend([x.text for x in lsit_5])
    
    links_pdf = driver.find_elements(By.XPATH, "//span[contains(@onclick,'descargar')]")
    
    dataframe_table = pd.DataFrame({'nro_resoluciones': nro_resoluciones,
                                     'fecha_resoluciones': fecha_resoluciones,
                                     'exp_resoluciones': exp_resoluciones,
                                     'denuncias': denuncias,
                                     'summillas': summillas})
    
    for link in links_pdf:
        time.sleep(random.randint(2, 3))
        link.click()    
     
    data_full[f'page_date_{i}'] = dataframe_table 
        
    driver.find_element(By.XPATH, "//*[@id='FormListado3:testpList:j_id114:9:cmlPage']").click()

    time.sleep(random.randint(5, 6))
    


100%|██████████| 7/7 [00:42<00:00,  6.07s/it]


In [ ]:
# //*[@id="FormListado3:testpList:j_id114:9:cmlPage"]

In [16]:
ResIndecopi = pd.concat( data_full.values() ).reset_index( drop = True )
ResIndecopi

,nro_resoluciones,fecha_resoluciones,exp_resoluciones,denuncias,summillas
0,3-2026/PS2,2026-01-05,003857-2025/PS2,"GUEVARA VASQUEZ, LUCY DORIS/RENTA CAR LIAM E.I...",>>ABANDONO >>ARCHIVO DEL PROCEDIMIENTO
1,4-2026/CPC-INDECOPI-PUN,2026-01-22,000022-2025/CPC-INDECOPI-PUN,DENUNCIANTE: JAVIER AMBROSIO PADILLA DIAZ / DE...,Declarar infundada en parte la denuncia interp...
2,8-2026/CPC-INDECOPI-CHT,2026-02-02,000068-2025/CPC-INDECOPI-CHT,DENUNCIANTE : ZENAIDA PAOLA MAGUIÑA FALCON DEN...,SEGUNDO: Denegar la solicitud de suspensión de...
3,9-2026/CPC-INDECOPI-CHT,2026-02-02,000074-2025/CPC-INDECOPI-CHT,DENUNCIANTE : GEREMIAS MATOS EGUSQUIZA DENUNCI...,PRIMERO: Declarar la confidencialidad de los A...
4,12-2026/CPC-INDECOPI-PUN,2026-02-26,000054-2025/CPC-INDECOPI-PUN,DENUNCIANTE: ELOÍSA ARCILA LUQUE CAYO / DENUNC...,Declarar fundado el procedimiento iniciado por...
...,...,...,...,...,...
166,2156-2026/PS2,2026-07-24,2480-2026/PS2,"DENUNCIANTES: ZEGARRA OTINIANO, CARLOS OSWALDO...",REGISTRO: 2480-2026/PS2 DENUNCIA - RESOLUCION ...
167,2231-2026/PS2,2026-08-05,2736-2026/PS2,"DENUNCIANTES: RAMIREZ GALVEZ, MARCO ANTONIO - ...",REGISTRO: 2736-2026/PS2 DENUNCIA - RESOLUCION ...
168,2243-2026/PS2,2026-08-07,2925-2026/PS2,"DENUNCIANTES: MASCO URRUTIA, JAIME - DENUNCIAD...",REGISTRO: 2925-2026/PS2 DENUNCIA - RESOLUCION ...
169,2265-2026/PS2,2026-08-11,2932-2026/PS2,"DENUNCIANTES: YUJCRE RIOS, JAIME ANTONIO - DEN...",REGISTRO: 2932-2026/PS2 DENUNCIA - RESOLUCION ...


In [ ]:
ResIndecopi.to_csv(r"..\..\data\raw\ResIndecopi.csv", 
                   index=False)

### Load and cleaning dataset

In [7]:
ResIndecopi = pd.read_csv(r"..\..\data\raw\ResIndecopi.csv",
                           encoding='utf-8')

In [8]:
ResIndecopi

,nro_resoluciones,fecha_resoluciones,exp_resoluciones,denuncias,summillas
0,3-2026/PS2,2026-01-05,003857-2025/PS2,"GUEVARA VASQUEZ, LUCY DORIS/RENTA CAR LIAM E.I...",>>ABANDONO >>ARCHIVO DEL PROCEDIMIENTO
1,4-2026/CPC-INDECOPI-PUN,2026-01-22,000022-2025/CPC-INDECOPI-PUN,DENUNCIANTE: JAVIER AMBROSIO PADILLA DIAZ / DE...,Declarar infundada en parte la denuncia interp...
2,8-2026/CPC-INDECOPI-CHT,2026-02-02,000068-2025/CPC-INDECOPI-CHT,DENUNCIANTE : ZENAIDA PAOLA MAGUIÑA FALCON DEN...,SEGUNDO: Denegar la solicitud de suspensión de...
3,9-2026/CPC-INDECOPI-CHT,2026-02-02,000074-2025/CPC-INDECOPI-CHT,DENUNCIANTE : GEREMIAS MATOS EGUSQUIZA DENUNCI...,PRIMERO: Declarar la confidencialidad de los A...
4,12-2026/CPC-INDECOPI-PUN,2026-02-26,000054-2025/CPC-INDECOPI-PUN,DENUNCIANTE: ELOÍSA ARCILA LUQUE CAYO / DENUNC...,Declarar fundado el procedimiento iniciado por...
...,...,...,...,...,...
166,2156-2026/PS2,2026-07-24,2480-2026/PS2,"DENUNCIANTES: ZEGARRA OTINIANO, CARLOS OSWALDO...",REGISTRO: 2480-2026/PS2 DENUNCIA - RESOLUCION ...
167,2231-2026/PS2,2026-08-05,2736-2026/PS2,"DENUNCIANTES: RAMIREZ GALVEZ, MARCO ANTONIO - ...",REGISTRO: 2736-2026/PS2 DENUNCIA - RESOLUCION ...
168,2243-2026/PS2,2026-08-07,2925-2026/PS2,"DENUNCIANTES: MASCO URRUTIA, JAIME - DENUNCIAD...",REGISTRO: 2925-2026/PS2 DENUNCIA - RESOLUCION ...
169,2265-2026/PS2,2026-08-11,2932-2026/PS2,"DENUNCIANTES: YUJCRE RIOS, JAIME ANTONIO - DEN...",REGISTRO: 2932-2026/PS2 DENUNCIA - RESOLUCION ...


In [9]:
ResIndecopi['EntidadFinanciera'] = ResIndecopi['denuncias'].apply(lambda x: quitar_tildes(re.split(r'[;:/]', x)[-1].strip()).lower())

In [11]:
ResIndecopi['EntidadFinanciera'].value_counts()

EntidadFinanciera
banco de credito del peru                                                                          33
banco interamericano de finanzas                                                                   18
banco de credito del peru s.a.                                                                     16
infinance xp s.a.                                                                                  12
banco bbva peru                                                                                    10
banco ripley peru s.a.                                                                              9
banco internacional del peru-interbank                                                              8
scotiabank peru saa                                                                                 8
caja municipal de ahorro y credito cusco s.a. – cmac cusco s.a.                                     4
banco de la nacion                                              

In [21]:
ResIndecopi['summillas'].to_list()

['>>ABANDONO >>ARCHIVO DEL PROCEDIMIENTO',
 'Declarar infundada en parte la denuncia interpuesta por el señor Javier Ambrosio Padilla Diaz, en contra de Scotiabank Perú Sociedad Anónima Abierta, por infracción al artículo 19º del Código de Prote... Ver más contenido.',
 'SEGUNDO: Denegar la solicitud de suspensión de procedimiento formulada por el Banco de Crédito del Perú S.A., conforme lo analizado en la presente resolución. TERCERO: Denegar la solicitud de inclusió... Ver más contenido.',
 'PRIMERO: Declarar la confidencialidad de los Anexos 2 y 3, así como de aquellas secciones del escrito del 27 de octubre de 2025, por constituir datos personales cuya reserva ha sido reconocida por ley... Ver más contenido.',
 'Declarar fundado el procedimiento iniciado por la señora Eloísa Arcila Luque Cayo en contra del Banco Internacional del Perú Sociedad Anónima Abierta - Interbank, por infracción al artículo 19° del Có... Ver más contenido.',
 'REGISTRO: 54-2025/CPC-PUN DENUNCIA - RESOLUCION

In [18]:
ResIndecopi.columns

Index(['nro_resoluciones', 'fecha_resoluciones', 'exp_resoluciones',
       'denuncias', 'summillas', 'EntidadFinanciera'],
      dtype='str')

In [22]:
ResIndecopi.to_csv(r"..\..\data\raw\ResIndecopi_metadata_pdf.csv", 
                   index=False)